In [ ]:
!pip install -qU transformers accelerate datasets optimum
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00


In [ ]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
import json
import os
from typing import Literal, List
from pydantic import BaseModel, Field
import random

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive"
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

dataset_path = os.path.join(base_path, "/content/drive/MyDrive/customer_questions.jsonl")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def read_dataset(path):
  input_data = []
  for line in open(dataset_path, "r"):
      if line.strip() == "":
        continue

      input_data.append(json.loads(line))
      random.Random(42).shuffle(input_data)
  print(f"Number of samples: {len(input_data)}")
  print(f"First sample: {input_data[0]}")
  print(f"Second sample: {input_data[1]}")
  return input_data

dataset = read_dataset(dataset_path)

Number of samples: 1674
First sample: {'question': 'هل تم شحن طلبي رقم 99887766 الذي يتضمن جهاز لابتوب، ومتى سيصل؟', 'classification': 'complex', 'intent': 'ask_about_order', 'products': [{'productName': 'لابتوب'}], 'order_numbers': ['99887766']}
Second sample: {'question': 'أنا عاوز أعرف طلبي رقم 654321987 فين دلوقتي وهيوصل امتى بالظبط؟', 'classification': 'complex', 'intent': 'ask_about_order', 'products': [], 'order_numbers': ['654321987']}


In [ ]:
Classification = Literal["simple", "complex", "complicated"]

Intent = Literal[
    "search_products",
    "ask_price",
    "compare_products",
    "ask_about_order",
    "ask_shipping",
    "ask_return",
    "ask_warranty",
    "product_availability",
    "product_specifications",
    "other"
]

class Product(BaseModel):
    productName: str = Field(..., description="The name of the product.")

class QuestionAnalysis(BaseModel):
    classification: Classification
    intent: Intent
    products: List[Product] = Field(
        default_factory=list,
        description="List of detected products."
    )
    order_numbers: List[str] = Field(
        default_factory=list,
        description="List of detected order numbers."
    )

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def get_message(quition: str):
    return  [
        {
          "role": "system",
           "content": (
               "Extract structured data."
               "classification is one of [simple, complex, complicated]."
               "intent is one of [search_products, ask_price, compare_products, ask_about_order, ask_shipping, ask_return, ask_warranty, product_availability, product_specifications, other]."
                "Return only JSON with keys: classification, intent, products, order_numbers."
           f"{QuestionAnalysis.schema_json()}"
           )
        },
        {
            "role": "user",
            "content": quition
        },
        {
            "role": "assistant",
            "content":"```JSON"
        }
    ]

In [ ]:
user_question =get_message("كم سعر تليفون ايفون 15 و وهاتف الترا اس5")

inputs = tokenizer.apply_chat_template(
    user_question,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=200)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("Raw model output:")
print(response)

/tmp/ipython-input-169/1159807904.py:10: PydanticDeprecatedSince20: The `schema_json` method is deprecated; use `model_json_schema` and json.dumps instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  f"{QuestionAnalysis.schema_json()}"


Raw model output:
{
  "classification": "simple",
  "intent": "search_products",
  "products": [
    {
      "productName": "iPhone 15"
    },
    {
      "productName": "iPhone 5T"
    }
  ],
  "order_numbers": []
}
```


##FineTune

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 629, done.
remote: Counting objects: 100% (629/629), done.
remote: Compressing objects: 100% (467/467), done.
remote: Total 629 (delta 151), reused 424 (delta 104), pack-reused 0 (from 0)
Receiving objects: 100% (629/629), 5.25 MiB | 13.15 MiB/s, done.
Resolving deltas: 100% (151/151), done.

Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

-e option requires 1 argument


In [ ]:
Intent

typing.Literal['search_products', 'ask_price', 'compare_products', 'ask_about_order', 'ask_shipping', 'ask_return', 'ask_warranty', 'product_availability', 'product_specifications', 'other']

In [ ]:
dataset[0]

{'question': 'هل تم شحن طلبي رقم 99887766 الذي يتضمن جهاز لابتوب، ومتى سيصل؟',
 'classification': 'complex',
 'intent': 'ask_about_order',
 'products': [{'productName': 'لابتوب'}],
 'order_numbers': ['99887766']}

In [ ]:
fine_dataset = []

system_message = "You are Classifier. You are NER System."

schema_hint = "{ classification, intent, products: [{productName}], order_numbers: [] }"

task = "\n".join([
    "classification: simple | complex | complicated",
    f"intent: {' | '.join(Intent.__args__)}",
    f"schema: {schema_hint}",
    "Return only JSON."
])

for data in dataset:
    output_json = json.dumps({
        "classification": data["classification"],
        "intent": data["intent"],
        "products": data["products"],
        "order_numbers": data["order_numbers"]
    }, ensure_ascii=False)

    fine_dataset.append({
        "system": system_message,
        "instruction": "\n".join([
            f"# Question: {data['question']}",
            f"# Task:\n{task}",
            "# Output JSON:",
            "```json"
        ]),
        "input": "",
        "output": f"{output_json}\n```",
    })

random.Random(42).shuffle(fine_dataset)
fine_dataset[15]

{'system': 'You are Classifier. You are NER System.',
 'instruction': '# Question: مقارنة بين لابتوب Lenovo Legion Pro 7i و Asus ROG Strix Scar 18 في الأداء الرسومي وسرعة المعالج.\n# Task:\nclassification: simple | complex | complicated\nintent: search_products | ask_price | compare_products | ask_about_order | ask_shipping | ask_return | ask_warranty | product_availability | product_specifications | other\nschema: { classification, intent, products: [{productName}], order_numbers: [] }\nReturn only JSON.\n# Output JSON:\n```json',
 'input': '',
 'output': '{"classification": "complex", "intent": "compare_products", "products": [{"productName": "Lenovo Legion Pro 7i"}, {"productName": "Asus ROG Strix Scar 18"}], "order_numbers": []}\n```'}

In [ ]:
train_size = len(fine_dataset)

train_dataset = fine_dataset[:int(train_size * 0.9)]
eval_dataset = fine_dataset[int(train_size * 0.9):]

print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

Train size: 1431
Eval size: 159


In [ ]:
import os
import json

fine_tune_dir = os.path.join(base_path, "fine_tune")
os.makedirs(fine_tune_dir, exist_ok=True)

with open(os.path.join(fine_tune_dir, "train.json"), "w", encoding="utf-8") as f:
    json.dump(train_dataset, f, ensure_ascii=False, default=str)

with open(os.path.join(fine_tune_dir, "eval.json"), "w", encoding="utf-8") as f:
    json.dump(eval_dataset, f, ensure_ascii=False, default=str)

In [ ]:
"""
"question_analysis_train": {
  "file_name": "/content/drive/MyDrive/fine_tune/train.json",
  "columns": {
    "prompt": "instruction",
    "query": "input",
    "response": "output",
    "system": "system",
    "history": "history"
  }
},

"question_analysis_eval": {
  "file_name": "/content/drive/MyDrive/fine_tune/eval.json",
  "columns": {
    "prompt": "instruction",
    "query": "input",
    "response": "output",
    "system": "system",
    "history": "history"
  }
}
"""

'\n"question_analysis_train": {\n  "file_name": "/content/drive/MyDrive/fine_tune/train.json",\n  "columns": {\n    "prompt": "instruction",\n    "query": "input",\n    "response": "output",\n    "system": "system",\n    "history": "history"\n  }\n},\n\n"question_analysis_eval": {\n  "file_name": "/content/drive/MyDrive/fine_tune/eval.json",\n  "columns": {\n    "prompt": "instruction",\n    "query": "input",\n    "response": "output",\n    "system": "system",\n    "history": "history"\n  }\n}\n'

In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_full/question_analysis.yaml
### model
model_name_or_path: "Qwen/Qwen2.5-0.5B-Instruct"
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: full

### dataset
dataset: question_analysis_train
template: qwen
cutoff_len: 100
#max_samples: 1000

preprocessing_num_workers: 16
dataloader_num_workers: 4
overwrite_cache: true

### output
#resume_from_checkpoint: /content/drive/MyDrive/fine_tune/model/sft
output_dir: /content/drive/MyDrive/fine_tune/model/sft
logging_steps: 10
save_steps: 25
plot_loss: true
overwrite_output_dir: true
save_only_model: false
report_to: none  # choices: [none, wandb, tensorboard, swanlab, mlflow]

### train
per_device_train_batch_size: 16
gradient_accumulation_steps: 2
learning_rate: 1.0e-4
num_train_epochs: 5.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000
resume_from_checkpoint: null

### eval
eval_dataset: question_analysis_eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 25


Overwriting /content/LLaMA-Factory/examples/train_full/question_analysis.yaml


In [ ]:
!pip install llamafactory

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.5/398.5 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!cd LLaMA-Factory && llamafactory-cli train examples/train_full/question_analysis.yaml

2026-02-27 08:42:49.006089: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772181769.027250    2618 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772181769.035125    2618 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772181769.052638    2618 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772181769.052661    2618 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772181769.052664    2618 computation_placer.cc:177] computation placer alr